# QLoRA fine-tune: Qwen3-4B → APUSH item writer (M3)

Distills the frontier-teacher dataset (`data/generated/train_sft.jsonl`, 816 audited items) into the small model, training **only on the JSON response** (the prompt is masked).

**Runtime:** Colab GPU — a free **T4** is enough for a 4B QLoRA. Set `Runtime → Change runtime type → GPU`.

**Data:** clones the repo to read `train_sft.jsonl`. If the repo is private, use the upload fallback cell.

Pipeline: install → load 4-bit model → add LoRA → load+template data → train (loss on response) → sanity-generate → save adapter (+ optional GGUF for Ollama).

In [ ]:
# 1) Install Unsloth (pulls compatible transformers/trl/peft/bitsandbytes).
%pip install -q unsloth
# keep bleeding-edge Unsloth in sync (recommended by Unsloth):
%pip install -q --upgrade --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

In [ ]:
# 2) Load Qwen3-4B in 4-bit. Swap model_name if you want a different Unsloth Qwen3 build.
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LEN = 4096          # prompt (~2.1k tok) + JSON completion fits comfortably
MODEL_NAME  = "unsloth/Qwen3-4B"   # matches the local Ollama candidate (qwen3:4b)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = MODEL_NAME,
    max_seq_length= MAX_SEQ_LEN,
    load_in_4bit  = True,
    dtype         = None,   # auto (bf16 on Ampere+, fp16 on T4)
)

In [ ]:
# 3) Attach LoRA adapters (QLoRA). r=16 is a solid default for a 4B on this data size.
model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"],
    lora_alpha     = 16,
    lora_dropout   = 0.0,
    bias           = "none",
    use_gradient_checkpointing = "unsloth",
    random_state   = 42,
)

In [ ]:
# 4) Get the dataset. Option A: clone the repo (public).
import os
if not os.path.exists("slm"):
    !git clone --depth 1 https://github.com/RohanPalivela/slm.git
DATA = "slm/data/generated/train_sft.jsonl"

# Option B (private repo): comment the clone above and upload the file instead:
# from google.colab import files; up = files.upload(); DATA = list(up.keys())[0]

print("dataset:", DATA, "exists:", os.path.exists(DATA))

In [ ]:
# 5) Apply the Qwen3 chat template to each example -> a single `text` field.
#    enable_thinking=False: we distill direct JSON output, no <think> traces.
from datasets import load_dataset

def to_text(ex):
    return {"text": tokenizer.apply_chat_template(
        ex["messages"], tokenize=False, add_generation_prompt=False, enable_thinking=False)}

dataset = load_dataset("json", data_files=DATA, split="train").map(to_text)
print(dataset)
print(dataset[0]["text"][:600])

In [ ]:
# 6) Trainer. train_on_responses_only masks the prompt so LOSS IS ON THE JSON ONLY.
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field          = "text",
        max_seq_length              = MAX_SEQ_LEN,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,      # effective batch 8
        warmup_steps                = 5,
        num_train_epochs            = 2,      # 816 ex * 2 ~= 200 steps; bump to 3 if underfit
        learning_rate               = 2e-4,
        logging_steps               = 5,
        optim                       = "adamw_8bit",
        weight_decay                = 0.01,
        lr_scheduler_type           = "linear",
        seed                        = 42,
        output_dir                  = "outputs",
        report_to                   = "none",
    ),
)

# Qwen3 uses ChatML turn markers.
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part    = "<|im_start|>assistant\n",
)

In [ ]:
# 7) Train.
stats = trainer.train()
print(stats)

In [ ]:
# 8) Sanity-generate on a HELD-OUT source (not in TRAIN) to eyeball the tuned output.
FastLanguageModel.for_inference(model)

SYSTEM = open("slm/prompts/litmus_generation_prompt.md").read()  # or paste your SYSTEM block
# Minimal user prompt for a quick check (use eval/prompt_loader for the exact format):
source = ("Federalist No. 10 ... the tendency to break and control the violence of faction ...")
user = (f'SOURCE:\n"""\n{source}\n"""\nWrite 1 APUSH stimulus-based MCQ. '
        'Requested archetype: CAUSE_OF_SOURCE. Return only the JSON array.')
msgs = [{"role": "system", "content": "You are a senior APUSH item writer. Return ONLY a JSON array."},
        {"role": "user", "content": user}]
ids = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True,
                                    enable_thinking=False, return_tensors="pt").to("cuda")
out = model.generate(input_ids=ids, max_new_tokens=512, temperature=0.7, do_sample=True)
print(tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True))

In [ ]:
# 9) Save the LoRA adapter (small, ~100MB). Download or push to the Hub.
model.save_pretrained("qwen3_4b_apush_lora")
tokenizer.save_pretrained("qwen3_4b_apush_lora")

# Optional: push to Hugging Face (needs a token)
# from huggingface_hub import login; login()
# model.push_to_hub("<your-username>/qwen3-4b-apush-lora", token=True)

# Optional: export a GGUF to run the tuned model in Ollama for base-vs-tuned eval
# model.save_pretrained_gguf("qwen3_4b_apush_gguf", tokenizer, quantization_method="q4_k_m")

## Next: base-vs-tuned eval

1. Export the tuned model to GGUF (cell 9) and register it in Ollama, **or** serve it via an OpenAI-compatible endpoint.
2. Add it to `eval/models.json` as a second `candidate` alongside base `qwen3:4b`.
3. Grow `EVAL_HELDOUT` to ~25–30 disjoint sources, then run the harness on that split and compare **near-miss (expert-quality)** rates: tuned > base is the win.

```bash
python eval/harness.py --split EVAL_HELDOUT --runs 3 --n 6
```